In [ ]:
from pybaseball import statcast
import pandas as pd
import datetime
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = statcast(start_dt='2026-03-25', end_dt='2026-10-10')

In [ ]:
df.info()

In [ ]:
dt = df[df['events'].isin(['field_out', 'force_out', 'single', 'double', 'strikeout', 'home_run', 'grounded_into_double_play', 'triple', 'fielders_choice_out', 'double_play', 'field_error', 'fielders_choice', 'strikeout_double_play', 'walk', 'hit_by_pitch']) == True][['game_date', 'batter', 'events', 'p_throws', 'home_team', 'away_team']]
dt = dt[['game_date', 'batter', 'events', 'p_throws']]
dt['game_date'] = pd.to_datetime(
    dt['game_date']
)
dt.head()

## WAVE

In [ ]:
latest = max(dt['game_date'])

sev_days_ago = latest - datetime.timedelta(days=81)

sev = dt[dt['game_date'] >= sev_days_ago]
def  hit(df):
    if df['events'] == 'single':
        return 1
    elif df['events'] == 'double':
        return 1
    elif df['events'] == 'triple':
        return 1
    elif df['events'] == 'home_run':
        return 1
    else:
        return 0
sev['hit'] = sev.apply(hit, axis=1)
lsev = sev[sev['p_throws'] == "L"].drop('p_throws', axis=1)
lsev['at_bat_lsev'] = 1
lsev = lsev[['batter', 'hit', 'at_bat_lsev']].groupby('batter', as_index = False).sum()
lsev['lsev_avg'] = lsev['hit'] / lsev['at_bat_lsev']
lsev = lsev[['batter', 'lsev_avg', 'at_bat_lsev']]
lsev.head()

In [ ]:
rsev = sev[sev['p_throws'] == "R"].drop('p_throws', axis=1)
rsev['at_bat_rsev'] = 1
rsev = rsev[['batter', 'hit', 'at_bat_rsev']].groupby('batter', as_index = False).sum()
rsev['rsev_avg'] = rsev['hit'] / rsev['at_bat_rsev']
rsev = rsev[['batter', 'rsev_avg', 'at_bat_rsev']]
rsev.head()

In [ ]:
fifteen_days_ago = latest - datetime.timedelta(days=10)

fif = dt[dt['game_date'] >= fifteen_days_ago]
fif['hit'] = fif.apply(hit, axis=1)
lfif = fif[fif['p_throws'] == "L"].drop('p_throws', axis=1)
lfif['at_bat_lfif'] = 1
lfif = lfif[['batter', 'hit', 'at_bat_lfif']].groupby('batter', as_index = False).sum()
lfif['lfif_avg'] = lfif['hit'] / lfif['at_bat_lfif']
lfif = lfif[['batter', 'lfif_avg', 'at_bat_lfif']]
lfif.head()

In [ ]:
rfif = fif[fif['p_throws'] == "R"].drop('p_throws', axis=1)
rfif['at_bat_rfif'] = 1
rfif = rfif[['batter', 'hit', 'at_bat_rfif']].groupby('batter', as_index = False).sum()
rfif['rfif_avg'] = rfif['hit'] / rfif['at_bat_rfif']
rfif = rfif[['batter', 'rfif_avg', 'at_bat_rfif']]
rfif.head()

In [ ]:
thirty_days_ago = latest - datetime.timedelta(days=30)

thir = dt[dt['game_date'] >= thirty_days_ago]
thir['hit'] = thir.apply(hit, axis=1)
lthir = thir[thir['p_throws'] == "L"].drop('p_throws', axis=1)
lthir['at_bat_lthir'] = 1
lthir = lthir[['batter', 'hit', 'at_bat_lthir']].groupby('batter', as_index = False).sum()
lthir['lthir_avg'] = lthir['hit'] / lthir['at_bat_lthir']
lthir = lthir[['batter', 'lthir_avg', 'at_bat_lthir']]
lthir.head()

In [ ]:
rthir = thir[thir['p_throws'] == "R"].drop('p_throws', axis=1)
rthir['at_bat_rthir'] = 1
rthir = rthir[['batter', 'hit', 'at_bat_rthir']].groupby('batter', as_index = False).sum()
rthir['rthir_avg'] = rthir['hit'] / rthir['at_bat_rthir']
rthir = rthir[['batter', 'rthir_avg', 'at_bat_rthir']]
rthir.head()

In [ ]:
full = dt
full['hit'] = full.apply(hit, axis=1)
lfull = full[full['p_throws'] == "L"].drop('p_throws', axis=1)
lfull['l_at_bat'] = 1
lfull = lfull[['batter', 'hit', 'l_at_bat']].groupby('batter', as_index = False).sum()
lfull['lfull_avg'] = lfull['hit'] / lfull['l_at_bat']
lfull = lfull[['batter', 'lfull_avg', 'l_at_bat']]
lfull.head()

In [ ]:
rfull = full[full['p_throws'] == "R"].drop('p_throws', axis=1)
rfull['r_at_bat'] = 1
rfull = rfull[['batter', 'hit', 'r_at_bat']].groupby('batter', as_index = False).sum()
rfull['rfull_avg'] = rfull['hit'] / rfull['r_at_bat']
rfull = rfull[['batter', 'rfull_avg', 'r_at_bat']]
rfull.head()

In [ ]:
wave = rfull.merge(lsev, on='batter', how='left').merge(lthir, on='batter', how='left').merge(lfif, on='batter', how='left').merge(lfull, on='batter', how='left').merge(rsev, on='batter', how='left').merge(rthir, on='batter', how='left').merge(rfif, on='batter', how='left')
from pybaseball import chadwick_register
names = chadwick_register()

wave =wave.rename(columns = {'batter' : 'key_mlbam'}).merge(names, on='key_mlbam', how='left')
wave = wave.drop(['key_retro', 'key_bbref', 'key_fangraphs', 'mlb_played_first', 'mlb_played_last'], axis=1)

wave.head()

In [ ]:
wave["abt"] = wave['l_at_bat'] + wave['r_at_bat']
wave['abtl'] = wave['l_at_bat'] / wave['abt']
wave['abtr'] = wave['r_at_bat'] / wave['abt']
wave['WAVE_R'] = wave['rfull_avg']*.15 + wave['rsev_avg']*.25 + wave['rthir_avg'] *.275 + wave['rfif_avg'] *.325
wave['WAVE_L'] = wave['lfull_avg'] *.15 + wave['lsev_avg'] *.25 + wave['lthir_avg'] *.275 + wave['lfif_avg'] *.325
wave['WAVE'] = wave['WAVE_R']*wave['abtr'] + wave['WAVE_L']*wave['abtl']
WAVE = wave[['name_first', 'name_last', 'WAVE', 'WAVE_L', 'WAVE_R', 'l_at_bat', 'r_at_bat', 'lsev_avg', 'rsev_avg', 'key_mlbam']]
WAVE['probability'] = 1 - ((1-WAVE['WAVE'])**3.5)
WAVE['probability_L'] = 1 - ((1-WAVE['WAVE_L'])**3.5)
WAVE['probability_R'] = 1 - ((1-WAVE['WAVE_R'])**3.5)
WAVE[(WAVE['l_at_bat'] + WAVE['r_at_bat']) > 30].nlargest(30, 'probability')

In [ ]:
WAVE[WAVE['name_first'] == "Brendan"].head()

## P_WAVE

In [ ]:
pdf = df[df['events'].isin(['field_out', 'force_out', 'single', 'double', 'strikeout', 'home_run', 'grounded_into_double_play', 'triple', 'fielders_choice_out', 'double_play', 'field_error', 'fielders_choice', 'strikeout_double_play', 'walk', 'hit_by_pitch']) == True][['game_date', 'pitcher', 'events']]
pdf['game_date'] = pd.to_datetime(
    pdf['game_date']
)
pdf.head()

In [ ]:
latest = max(dt['game_date'])

sev_days_ago = latest - datetime.timedelta(days=15)

sev = pdf[pdf['game_date'] >= sev_days_ago]
def  hit(df):
    if df['events'] == 'single':
        return 1
    elif df['events'] == 'double':
        return 1
    elif df['events'] == 'triple':
        return 1
    elif df['events'] == 'home_run':
        return 1
    else:
        return 0

def  bases(df):
    if df['events'] == 'single':
        return 1
    elif df['events'] == 'double':
        return 2
    elif df['events'] == 'triple':
        return 3
    elif df['events'] == 'home_run':
        return 4
    else:
        return 0

def stk(df):
  if df['events'] == 'strikeout':
    return 1
  elif df['events'] == 'strikeout_double_play':
    return 1
  elif df['events'] == 'walk':
    return 1
  elif df['events'] == 'hit_by_pitch':
    return 1
  else:
    return 0

def out(df):
  if df['events'] == 'single':
    return 0
  elif df['events'] == 'double':
    return 0
  elif df['events'] == 'triple':
    return 0
  elif df['events'] == 'home_run':
    return 0
  elif df['events'] == 'walk':
    return 0
  elif df['events'] == 'hit_by_pitch':
    return 0
  else:
    return 1

def homer(df):
  if df['events'] == 'home_run':
    return 1
  else:
    return 0

sev['out'] = sev.apply(out, axis=1)

sev['hit'] = sev.apply(hit, axis=1)
sev['stk'] = sev.apply(stk, axis=1)
sev['TBA'] = sev.apply(bases, axis=1)
sev['hr'] = sev.apply(homer, axis=1)
sev['at_bat_sev'] = 1
sev = sev[['pitcher', 'hit', 'stk', 'TBA', 'hr','at_bat_sev']].groupby('pitcher', as_index = False).sum()
sev['baa_fif'] = sev['hit'] / sev['at_bat_sev']
sev['stk_per_sev'] = sev['stk'] / sev['at_bat_sev']
sev['pave_fif'] = sev['baa_fif']/(1-sev['stk_per_sev'])
sev['power_a_fif'] = sev['TBA']/sev['at_bat_sev']
sev['hr_fif'] = sev['hr'] / sev['at_bat_sev']
sev = sev[['pitcher', 'pave_fif', 'baa_fif', 'power_a_fif', 'hr_fif']]
sev.head()

In [ ]:
thir_days_ago = latest - datetime.timedelta(days=30)
thir = pdf[pdf['game_date'] >= thir_days_ago]
thir['hit'] = thir.apply(hit, axis=1)
thir['stk'] = thir.apply(stk, axis=1)
thir['TBA'] = thir.apply(bases, axis=1)
thir['hr'] = thir.apply(homer, axis=1)
thir['at_bat_thir'] = 1
thir = thir[['pitcher', 'hit', 'stk', 'TBA', 'hr', 'at_bat_thir']].groupby('pitcher', as_index = False).sum()
thir['baa_thir'] = thir['hit'] / thir['at_bat_thir']
thir['stk_per_thir'] = thir['stk'] / thir['at_bat_thir']
thir['pave_thir'] = thir['baa_thir']/(1-thir['stk_per_thir'])
thir['power_a_thir'] = thir['TBA']/thir['at_bat_thir']
thir['hr_thir'] = thir['hr'] / thir['at_bat_thir']
thir = thir[['pitcher', 'pave_thir', 'baa_thir', 'hr_thir', 'power_a_thir']]
thir.head()

In [ ]:
half_days_ago = latest - datetime.timedelta(days=81)
half = pdf[pdf['game_date'] >= half_days_ago]
half['hit'] = half.apply(hit, axis=1)
half['stk'] = half.apply(stk, axis=1)
half['TBA'] = half.apply(bases, axis=1)
half['hr'] = half.apply(homer, axis=1)
half['at_bat_half'] = 1
half = half[['pitcher', 'hit', 'stk', 'TBA', 'hr', 'at_bat_half']].groupby('pitcher', as_index = False).sum()
half['baa_half'] = half['hit'] / half['at_bat_half']
half['stk_per_half'] = half['stk'] / half['at_bat_half']
half['pave_half'] = half['baa_half']/(1-half['stk_per_half'])
half['power_a_half'] = half['TBA']/half['at_bat_half']
half['hr_half'] = half['hr'] / half['at_bat_half']
half = half[['pitcher', 'pave_half', 'baa_half', 'power_a_half', 'hr_half']]
half.head()

In [ ]:
pull = pdf
pull['hits'] = pull.apply(hit, axis=1)
pull['stk'] = pull.apply(stk, axis=1)
pull['TBA'] = pull.apply(bases, axis=1)
pull['hr'] = pull.apply(homer, axis=1)
pull['at_bats'] = 1
pull = pull[['pitcher', 'hits', 'stk', 'TBA', 'at_bats', 'hr']].groupby('pitcher', as_index = False).sum()
pull['baa_full'] = pull['hits'] / pull['at_bats']
pull['stk_per_full'] = pull['stk'] / pull['at_bats']
pull['pave_full'] = pull['baa_full']/(1-pull['stk_per_full'])
pull['power_a_full'] = pull['TBA']/pull['at_bats']
pull['hr_full'] = pull['hr'] / pull['at_bats']
pull = pull[['pitcher', 'pave_full', 'baa_full', 'power_a_full', 'hr_full', 'at_bats', 'hits', 'TBA']]
pull.head()

In [ ]:
pave = pull.merge(thir, on='pitcher', how='left').merge(sev, on='pitcher', how='left').merge(half, on = 'pitcher', how = 'left')
pave['PAVE'] = pave['pave_full'].fillna(0) * .3 + pave['pave_thir'].fillna(0) * .265 + pave['pave_half'].fillna(0)*.23 +pave['pave_fif'].fillna(0) * .205
pave['power_a'] = pave['power_a_full'].fillna(0) * .3 + pave['power_a_thir'].fillna(0) * .265 + pave['power_a_half'].fillna(0) * .23 + pave['power_a_fif'].fillna(0) * .205
pave['baa'] = pave['baa_full'].fillna(0) * .3 + pave['baa_thir'].fillna(0) * .265 + pave['baa_half'].fillna(0) * .23 + pave['baa_fif'].fillna(0) * .205
pave['hr_per'] = pave['hr_full'].fillna(0) * .3 + pave['hr_thir'].fillna(0) * .265 + pave['hr_half'].fillna(0) * .23 + pave['hr_fif'].fillna(0) * .205

from pybaseball import chadwick_register
names = chadwick_register()

pave =pave.rename(columns = {'pitcher' : 'key_mlbam'}).merge(names, on='key_mlbam', how='left')
pave = pave.drop(['key_mlbam', 'key_retro', 'key_bbref', 'key_fangraphs', 'mlb_played_first', 'mlb_played_last'], axis=1)

pave = pave[['name_first', 'name_last', 'PAVE', 'at_bats', 'hits', 'TBA', 'pave_fif', 'pave_thir', 'pave_half', 'pave_full', 'power_a', 'baa', 'hr_per']]

In [ ]:
qual = pave['at_bats'].max() * .75
mean_p = pave[pave['at_bats'] > qual]['PAVE'].mean()
pave['PAVE_PLUS'] = pave['PAVE']/mean_p
pave['PAVE_PLUS'] = pave['PAVE_PLUS'].fillna(0)
pave = pave[pave['PAVE_PLUS'] > 0]
quat = pave['at_bats'].max() * .25
filt = pave[pave['at_bats'] > quat]['at_bats'].mean()
pave = pave[pave['at_bats'] > filt]
pave['Expected_Hits'] = pave['baa'] * 22
pave['Expected_Bases'] = pave['power_a'] * 22
pave['Expected_HRs'] = pave['hr_per'] * 22
pave = pave[['name_first', 'name_last', 'at_bats', 'PAVE_PLUS', 'Expected_Hits', 'Expected_Bases', 'Expected_HRs']]
pave = pave.sort_values('PAVE_PLUS', ascending=False)
pave[pave['at_bats'] > 30].nlargest(20, 'Expected_Bases')

In [ ]:
pave[pave['name_last'] == 'Smith'].head()

# Hit_Prob

In [ ]:
gam_id = df[["game_date", "home_team", "away_team", "pitcher", "at_bat_number"]].drop_duplicates().iloc[::-1]

def create_id(df, col_check):
  count = 0
  new_col = []
  for val in df[col_check]:
      if val == 1:
          count += 1
      new_col.append(count)
  df['game_id'] = new_col
  return df

gam_id = create_id(gam_id, 'at_bat_number')

data = df.merge(gam_id, on = ["game_date", "home_team", "away_team", "pitcher", "at_bat_number"])
data['ind'] = (data['game_id'].astype('str') + data['at_bat_number'].astype('str') + data['pitch_number'].astype('str')).astype('int')
data = data.set_index('ind')
data = data.sort_index()
data[data['game_id'] == 8].head()

In [ ]:
dr = data[data['events'].isin(['field_out', 'force_out', 'single', 'double', 'strikeout', 'home_run', 'grounded_into_double_play', 'triple', 'fielders_choice_out', 'double_play', 'field_error', 'fielders_choice', 'strikeout_double_play', 'walk', 'hit_by_pitch']) == True]
dr = dr[['game_date', 'batter', 'events', 'game_id']]
dr.head()

In [ ]:
latest = max(dt['game_date'])

sev_days_ago = latest - datetime.timedelta(days=81)
hit_events = ['single', 'double', 'triple', 'home_run']

dr['had_hit'] = dr['events'].isin(hit_events).astype(int)

game_hits = (
    dr.groupby(['batter', 'game_id', 'game_date'])['had_hit']
    .max()
    .reset_index()
)
game_hits['game'] = 1
ps = game_hits[game_hits['game_date'] >= sev_days_ago][['batter', 'had_hit', 'game']]
ps = ps.groupby('batter', as_index = False).sum()
ps['hit_prob_s'] = ps['had_hit'] / ps['game']
ps = ps[['batter', 'hit_prob_s']]
ps.head()

In [ ]:
fifteen_days_ago = latest - datetime.timedelta(days=10)

pf = game_hits[game_hits['game_date'] >= fifteen_days_ago][['batter', 'had_hit', 'game']]
pf = pf.groupby('batter', as_index = False).sum()
pf['hit_prob_f'] = pf['had_hit'] / pf['game']
pf = pf[['batter', 'hit_prob_f']]
pf.head()

In [ ]:
thirty_days_ago = latest - datetime.timedelta(days=30)

pt = game_hits[game_hits['game_date'] >= thirty_days_ago][['batter', 'had_hit', 'game']]
pt = pt.groupby('batter', as_index = False).sum()
pt['hit_prob_t'] = pt['had_hit'] / pt['game']
pt = pt[['batter', 'hit_prob_t']]
pt.head()

In [ ]:
full = game_hits[['batter', 'had_hit', 'game']]
full = full.groupby('batter', as_index = False).sum()
full['hit_prob'] = full['had_hit'] / full['game']
full = full[['batter', 'hit_prob']]
full = full.merge(ps, on='batter', how='left').merge(pf, on='batter', how='left').merge(pt, on='batter', how='left').fillna(0)
full['Game_Hit_Probability'] = full['hit_prob'] * .175 + full['hit_prob_s'] * .225 + full['hit_prob_t'] * .275 + full['hit_prob_f'] * .325
full.head()

In [ ]:
prob =full.rename(columns = {'batter' : 'key_mlbam'})[['key_mlbam', 'Game_Hit_Probability']]
prob.head()

## WHOPS

In [ ]:
latest = max(dt['game_date'])

sev_days_ago = latest - datetime.timedelta(days=7)

sev = dt[dt['game_date'] >= sev_days_ago]
def  ob(df):
    if df['events'] == 'single':
        return 1
    elif df['events'] == 'double':
        return 1
    elif df['events'] == 'triple':
        return 1
    elif df['events'] == 'home_run':
        return 1
    elif df['events'] == 'walk':
        return 1
    elif df['events'] == 'hit_by_pitch':
        return 1
    else:
        return 0

def  ab(df):
    if df['events'] == 'walk':
        return 0
    elif df['events'] == 'hit_by_pitch':
        return 0
    else:
        return 1

def  sv(df):
    if df['events'] == 'single':
        return 1
    elif df['events'] == 'double':
        return 2
    elif df['events'] == 'triple':
        return 3
    elif df['events'] == 'home_run':
        return 4
    else:
        return 0

sev['ob'] = sev.apply(ob, axis=1)
sev['sv'] = sev.apply(sv, axis=1)
sev['ab'] = sev.apply(ab, axis=1)
lsev = sev[sev['p_throws'] == "L"].drop('p_throws', axis=1)
lsev['pa_lsev'] = 1
lsev = lsev[['batter', 'ob', 'sv', 'ab', 'pa_lsev']].groupby('batter', as_index = False).sum()
lsev['lsev_slg'] = lsev['sv'] / lsev['pa_lsev']
lsev['lsev_obp'] = lsev['ob'] / lsev['pa_lsev']
lsev['lsev_ops'] = lsev['lsev_obp'] + lsev['lsev_slg']
lsev['lsev_rc'] = lsev['lsev_slg'] * lsev['lsev_obp'] * lsev['ab']/lsev['pa_lsev']
lsev = lsev[['batter', 'lsev_ops', 'pa_lsev', 'lsev_rc']]
lsev.head()

In [ ]:
rsev = sev[sev['p_throws'] == "R"].drop('p_throws', axis=1)
rsev['pa_rsev'] = 1
rsev = rsev[['batter', 'ob', 'sv','pa_rsev', 'ab']].groupby('batter', as_index = False).sum()
rsev['rsev_slg'] = rsev['sv'] / rsev['pa_rsev']
rsev['rsev_obp'] = rsev['ob'] / rsev['pa_rsev']
rsev['rsev_ops'] = rsev['rsev_obp'] + rsev['rsev_slg']
rsev['rsev_rc'] = rsev['rsev_slg'] * rsev['rsev_obp'] * rsev['ab']/rsev['pa_rsev']
rsev = rsev[['batter', 'rsev_ops', 'pa_rsev', 'rsev_rc']]
rsev.head()

In [ ]:
fif_days_ago = latest - datetime.timedelta(days=15)
fif = dt[dt['game_date'] >= fif_days_ago]
fif['ob'] = fif.apply(ob, axis=1)
fif['sv'] = fif.apply(sv, axis=1)
fif['ab'] = fif.apply(ab, axis=1)
lfif = fif[fif['p_throws'] == "L"].drop('p_throws', axis=1)
lfif['pa_lfif'] = 1
lfif = lfif[['batter', 'ob', 'sv','pa_lfif', 'ab']].groupby('batter', as_index = False).sum()
lfif['lfif_slg'] = lfif['sv'] / lfif['pa_lfif']
lfif['lfif_obp'] = lfif['ob'] / lfif['pa_lfif']
lfif['lfif_ops'] = lfif['lfif_obp'] + lfif['lfif_slg']
lfif['lfif_rc'] = lfif['lfif_slg'] * lfif['lfif_obp'] * lfif['ab']/lfif['pa_lfif']
lfif = lfif[['batter', 'lfif_ops', 'pa_lfif', 'lfif_rc']]
lfif.head()

In [ ]:
rfif = fif[fif['p_throws'] == "R"].drop('p_throws', axis=1)
rfif['pa_rfif'] = 1
rfif = rfif[['batter', 'ob', 'sv','pa_rfif', 'ab']].groupby('batter', as_index = False).sum()
rfif['rfif_slg'] = rfif['sv'] / rfif['pa_rfif']
rfif['rfif_obp'] = rfif['ob'] / rfif['pa_rfif']
rfif['rfif_ops'] = rfif['rfif_obp'] + rfif['rfif_slg']
rfif['rfif_rc'] = rfif['rfif_slg'] * rfif['rfif_obp'] * rfif['ab']/rfif['pa_rfif']
rfif = rfif[['batter', 'rfif_ops', 'pa_rfif', 'rfif_rc']]
rfif.head()

In [ ]:
thir_days_ago = latest - datetime.timedelta(days=30)
thir = dt[dt['game_date'] >= thir_days_ago]
thir['ob'] = thir.apply(ob, axis=1)
thir['sv'] = thir.apply(sv, axis=1)
thir['ab'] = thir.apply(ab, axis=1)
lthir = thir[thir['p_throws'] == "L"].drop('p_throws', axis=1)
lthir['pa_lthir'] = 1
lthir = lthir[['batter', 'ob', 'sv','pa_lthir', 'ab']].groupby('batter', as_index = False).sum()
lthir['lthir_slg'] = lthir['sv'] / lthir['pa_lthir']
lthir['lthir_obp'] = lthir['ob'] / lthir['pa_lthir']
lthir['lthir_ops'] = lthir['lthir_obp'] + lthir['lthir_slg']
lthir['lthir_rc'] = lthir['lthir_slg'] * lthir['lthir_obp'] * lthir['ab']/lthir['pa_lthir']
lthir = lthir[['batter', 'lthir_ops', 'pa_lthir', 'lthir_rc']]
lthir.head()

In [ ]:
rthir = thir[thir['p_throws'] == "R"].drop('p_throws', axis=1)
rthir['pa_rthir'] = 1
rthir = rthir[['batter', 'ob', 'sv','pa_rthir', 'ab']].groupby('batter', as_index = False).sum()
rthir['rthir_slg'] = rthir['sv'] / rthir['pa_rthir']
rthir['rthir_obp'] = rthir['ob'] / rthir['pa_rthir']
rthir['rthir_ops'] = rthir['rthir_obp'] + rthir['rthir_slg']
rthir['rthir_rc'] = rthir['rthir_slg'] * rthir['rthir_obp'] * rthir['ab']/rthir['pa_rthir']
rthir = rthir[['batter', 'rthir_ops', 'pa_rthir', 'rthir_rc']]
rthir.head()

In [ ]:
full = dt
full['ob'] = full.apply(ob, axis=1)
full['sv'] = full.apply(sv, axis=1)
full['ab'] = full.apply(ab, axis=1)
lfull = full[full['p_throws'] == "L"].drop('p_throws', axis=1)
lfull['pa_lfull'] = 1
lfull = lfull[['batter', 'ob', 'sv','pa_lfull', 'ab']].groupby('batter', as_index = False).sum()
lfull['lfull_slg'] = lfull['sv'] / lfull['pa_lfull']
lfull['lfull_obp'] = lfull['ob'] / lfull['pa_lfull']
lfull['lfull_ops'] = lfull['lfull_obp'] + lfull['lfull_slg']
lfull['lfull_rc'] = lfull['lfull_slg'] * lfull['lfull_obp'] * lfull['ab']/lfull['pa_lfull']
lfull = lfull[['batter', 'lfull_ops', 'pa_lfull', 'lfull_rc']]
lfull.info()

In [ ]:
rfull = full[full['p_throws'] == "R"].drop('p_throws', axis=1)
rfull['pa_rfull'] = 1
rfull = rfull[['batter', 'ob', 'sv','pa_rfull', 'ab']].groupby('batter', as_index = False).sum()
rfull['rfull_slg'] = rfull['sv'] / rfull['pa_rfull']
rfull['rfull_obp'] = rfull['ob'] / rfull['pa_rfull']
rfull['rfull_ops'] = rfull['rfull_obp'] + rfull['rfull_slg']
rfull['rfull_rc'] = rfull['rfull_slg'] * rfull['rfull_obp'] * rfull['ab']/rfull['pa_rfull']
rfull = rfull[['batter', 'rfull_ops', 'pa_rfull', 'rfull_rc']]
rfull.info()

In [ ]:
whops = rfull.merge(lsev, on='batter', how='left').merge(lthir, on='batter', how='left').merge(lfif, on='batter', how='left').merge(lfull, on='batter', how='left').merge(rsev, on='batter', how='left').merge(rthir, on='batter', how='left').merge(rfif, on='batter', how='left')
from pybaseball import chadwick_register
names = chadwick_register()

whops =whops.rename(columns = {'batter' : 'key_mlbam'}).merge(names, on='key_mlbam', how='left')
whops = whops.drop(['key_mlbam', 'key_retro', 'key_bbref', 'key_fangraphs', 'mlb_played_first', 'mlb_played_last'], axis=1)

whops[whops['name_last'] == 'Langford'].head()

In [ ]:
whops['abt'] = whops['pa_lfull'] + whops['pa_rfull']
whops['abtl'] = whops['pa_lfull'] / whops['abt']
whops['abtr'] = whops['pa_rfull'] / whops['abt']
whops['whops_R'] = whops['rfull_ops']*.175 + whops['rsev_ops']*.225 + whops['rthir_ops']*.275 + whops['rfif_ops']*.325
whops['whops_L'] = whops['lfull_ops']*.175 + whops['lsev_ops']*.225 + whops['lthir_ops']*.275 + whops['lfif_ops']*.325
whops['whops'] = whops['whops_R']*whops['abtr'] + whops['whops_L']*whops['abtl']
whops['rcw_R'] = whops['rfull_rc']*.175 + whops['rsev_rc']*.225 + whops['rthir_rc']*.275 + whops['rfif_rc']*.325
whops['rcw_L'] = whops['lfull_rc']*.175 + whops['lsev_rc']*.225 + whops['lthir_rc']*.275 + whops['lfif_rc']*.325
whops['rcw'] = whops['rcw_R']*whops['abtr'] + whops['rcw_L']*whops['abtl']
whops['rope'] = whops['rcw'] + whops['whops']
WHOPS = whops[['name_first', 'name_last', 'whops', 'whops_R', 'whops_L', 'pa_lfull', 'pa_rfull', 'rcw_R', 'rcw_L', 'rcw', 'rope']]
WHOPS[(WHOPS['pa_lfull'] + WHOPS['pa_rfull']) > 30].nlargest(30, 'rope')

In [ ]:
WHOPS[WHOPS['name_last'] == "Rice" ].head()

# Weighted Bases

In [ ]:
latest = max(dt['game_date'])

sev_days_ago = latest - datetime.timedelta(days=7)

sev = dt[dt['game_date'] >= sev_days_ago]

def  sv(df):
    if df['events'] == 'single':
        return 1
    elif df['events'] == 'double':
        return 2
    elif df['events'] == 'triple':
        return 3
    elif df['events'] == 'home_run':
        return 4
    else:
        return 0

sev['sv'] = sev.apply(sv, axis=1)
lsev = sev[sev['p_throws'] == "L"].drop('p_throws', axis=1)
lsev['pa_lsev'] = 1
lsev = lsev[['batter', 'sv', 'pa_lsev']].groupby('batter', as_index = False).sum()
lsev['lsev_slg'] = lsev['sv'] / lsev['pa_lsev']
lsev = lsev[['batter', 'lsev_slg', 'pa_lsev']]
lsev.head()

In [ ]:
rsev = sev[sev['p_throws'] == "R"].drop('p_throws', axis=1)
rsev['pa_rsev'] = 1
rsev = rsev[['batter', 'sv','pa_rsev']].groupby('batter', as_index = False).sum()
rsev['rsev_slg'] = rsev['sv'] / rsev['pa_rsev']
rsev = rsev[['batter', 'rsev_slg', 'pa_rsev']]
rsev.head()

In [ ]:
fif_days_ago = latest - datetime.timedelta(days=15)
fif = dt[dt['game_date'] >= fif_days_ago]
fif['sv'] = fif.apply(sv, axis=1)
lfif = fif[fif['p_throws'] == "L"].drop('p_throws', axis=1)
lfif['pa_lfif'] = 1
lfif = lfif[['batter', 'sv','pa_lfif']].groupby('batter', as_index = False).sum()
lfif['lfif_slg'] = lfif['sv'] / lfif['pa_lfif']
lfif = lfif[['batter', 'lfif_slg', 'pa_lfif']]
lfif.head()

In [ ]:
rfif = fif[fif['p_throws'] == "R"].drop('p_throws', axis=1)
rfif['pa_rfif'] = 1
rfif = rfif[['batter', 'sv','pa_rfif']].groupby('batter', as_index = False).sum()
rfif['rfif_slg'] = rfif['sv'] / rfif['pa_rfif']
rfif = rfif[['batter', 'rfif_slg', 'pa_rfif']]
rfif.head()

In [ ]:
thir_days_ago = latest - datetime.timedelta(days=30)
thir = dt[dt['game_date'] >= thir_days_ago]
thir['sv'] = thir.apply(sv, axis=1)
lthir = thir[thir['p_throws'] == "L"].drop('p_throws', axis=1)
lthir['pa_lthir'] = 1
lthir = lthir[['batter', 'sv', 'pa_lthir']].groupby('batter', as_index = False).sum()
lthir['lthir_slg'] = lthir['sv'] / lthir['pa_lthir']
lthir = lthir[['batter', 'lthir_slg', 'pa_lthir']]
lthir.head()

In [ ]:
rthir = thir[thir['p_throws'] == "R"].drop('p_throws', axis=1)
rthir['pa_rthir'] = 1
rthir = rthir[['batter', 'ob', 'sv','pa_rthir', 'ab']].groupby('batter', as_index = False).sum()
rthir['rthir_slg'] = rthir['sv'] / rthir['pa_rthir']
rthir = rthir[['batter', 'rthir_slg', 'pa_rthir']]
rthir.head()

In [ ]:
full = dt
full['sv'] = full.apply(sv, axis=1)
lfull = full[full['p_throws'] == "L"].drop('p_throws', axis=1)
lfull['pa_lfull'] = 1
lfull = lfull[['batter', 'ob', 'sv','pa_lfull', 'ab']].groupby('batter', as_index = False).sum()
lfull['lfull_slg'] = lfull['sv'] / lfull['pa_lfull']
lfull = lfull[['batter', 'lfull_slg', 'pa_lfull']]
lfull.info()

In [ ]:
rfull = full[full['p_throws'] == "R"].drop('p_throws', axis=1)
rfull['pa_rfull'] = 1
rfull = rfull[['batter', 'sv','pa_rfull']].groupby('batter', as_index = False).sum()
rfull['rfull_slg'] = rfull['sv'] / rfull['pa_rfull']
rfull = rfull[['batter', 'rfull_slg', 'pa_rfull']]
rfull.info()

In [ ]:
wslg = rfull.merge(lsev, on='batter', how='left').merge(lthir, on='batter', how='left').merge(lfif, on='batter', how='left').merge(lfull, on='batter', how='left').merge(rsev, on='batter', how='left').merge(rthir, on='batter', how='left').merge(rfif, on='batter', how='left')
from pybaseball import chadwick_register
names = chadwick_register()

wslg =wslg.rename(columns = {'batter' : 'key_mlbam'}).merge(names, on='key_mlbam', how='left')
wslg = wslg.drop(['key_retro', 'key_bbref', 'key_fangraphs', 'mlb_played_first', 'mlb_played_last'], axis=1)

wslg[wslg['name_last'] == 'Langford'].head()

In [ ]:
wslg['ab'] = wslg['pa_lfull'] + wslg['pa_rfull']
wslg['abr'] = wslg['pa_rfull'] / wslg['ab']
wslg['abl'] = wslg['pa_lfull'] / wslg['ab']
wslg['WTB_r'] = wslg['rsev_slg']*.225 + wslg['rthir_slg']*.275 + wslg['rfif_slg']*.325 + wslg['rfull_slg']*.175
wslg['WTB_l'] = wslg['lsev_slg']*.225 + wslg['lthir_slg']*.275 + wslg['lfif_slg']*.325 + wslg['lfull_slg']*.175
wslg['WTB'] = wslg['WTB_r']*wslg['abr'] + wslg['WTB_l']*wslg['abl']
wslg['Expected_Bases'] = wslg['WTB']*3.5
WTB = wslg[['name_first', 'name_last', 'pa_lfull', 'pa_rfull', 'WTB_l', 'WTB_r', 'WTB', 'Expected_Bases', 'key_mlbam']]
WTB[WTB['pa_rfull'] + WTB['pa_lfull'] > 70].nlargest(30, 'Expected_Bases')

In [ ]:
WAVE = WAVE.drop(['name_first', 'name_last'], axis = 1)
WAVE = WAVE.merge(WTB, on='key_mlbam', how ='left').merge(prob, on = 'key_mlbam', how = 'left')
WAVE['Consistency'] = WAVE['Game_Hit_Probability'] - WAVE['probability']
WAVE['Approach'] = WAVE['Game_Hit_Probability'] * WAVE['probability']
WAVE = WAVE[['name_first', 'name_last', 'pa_lfull', 'pa_rfull', 'probability_L', 'probability_R', 'probability', 'Game_Hit_Probability', 'Consistency', 'Approach', 'Expected_Bases']].fillna(0)
WAVE = WAVE[WAVE['Consistency'] > 0]
WAVE = WAVE.sort_values('Game_Hit_Probability', ascending=False)
WAVE = WAVE.rename(columns = {'pa_lfull' : 'PA_L', 'pa_rfull' : 'PA_R'})
pa = WAVE['PA_L'] + WAVE['PA_R']
wall = pa.max() * .25
WAVE = WAVE[(WAVE['PA_L'] + WAVE['PA_R']) > wall]

# Look Ups

In [ ]:
WAVE[(WAVE['PA_L'] + WAVE['PA_R']) > 130].nlargest(30, 'Game_Hit_Probability')

In [ ]:
name = 'Antonacci'
WAVE[(WAVE['name_last'] == name) | (WAVE['name_first'] == name)].head().sort_values(by='probability', ascending=False)

In [ ]:
pave[pave['at_bats'] >150].nlargest(30, 'PAVE_PLUS')

In [ ]:
name = 'Flaherty'
pave[(pave['name_last'] == name) | (pave['name_first'] == name)].head()

#Strength

In [ ]:
gam_id = df[["game_date", "home_team", "away_team", "pitcher", "at_bat_number"]].drop_duplicates().iloc[::-1]

def create_id(df, col_check):
  count = 0
  new_col = []
  for val in df[col_check]:
      if val == 1:
          count += 1
      new_col.append(count)
  df['game_id'] = new_col
  return df

gam_id = create_id(gam_id, 'at_bat_number')

data = df.merge(gam_id, on = ["game_date", "home_team", "away_team", "pitcher", "at_bat_number"])
data['ind'] = (data['game_id'].astype('str') + data['at_bat_number'].astype('str') + data['pitch_number'].astype('str')).astype('int')
data = data.set_index('ind')
data = data.sort_index()

In [ ]:
hhr = data[data['inning_topbot'] == 'Bot'][['home_team', 'events', 'home_score', 'post_home_score', 'game_id']]
hhr = hhr.rename(columns = {'home_team' : 'team', 'home_score' : 'pre_score', 'post_home_score' : 'post_score'})
ahr = data[data['inning_topbot'] == 'Top'][['away_team', 'events', 'away_score', 'post_away_score', 'game_id']]
ahr = ahr.rename(columns = {'away_team' : 'team', 'away_score' : 'pre_score', 'post_away_score' : 'post_score'})
hr = pd.concat([hhr, ahr])

def homer(df):
  if df['events'] == 'home_run':
    return 'homer'
  else:
    return 'non_homer'

hr['hr'] = hr.apply(homer, axis = 1)

gp = hr[['team', 'game_id']].drop_duplicates()
gp = gp.groupby('team', as_index = False).count()
gp = gp.rename(columns = {'game_id' : 'game_count'})
total_homers = hr[['team', 'hr']][hr['hr'] == 'homer'].groupby('team', as_index = False).count()
hpg = total_homers.merge(gp, on = 'team')
hpg['homer_per_game'] = hpg['hr'] / hpg['game_count']
hpg = hpg[['team', 'homer_per_game']]
gwh = hr[['team', 'game_id', 'hr']][hr['hr'] == 'homer'].groupby(['team', 'game_id'], as_index = False).count()
game = hr[['team', 'game_id']].drop_duplicates()
gwh = game.merge(gwh, on = ['team', 'game_id'], how = 'left').fillna(0)
ghi = gwh[gwh['hr'] > 0].groupby('team', as_index = False).count()
ghi = ghi.rename(columns = {'hr' : 'games_homered_in'})
gnhi = gwh[gwh['hr'] == 0].groupby('team', as_index = False).count()
gnhi = gnhi.rename(columns = {'hr' : 'games_not_homered_in'})
ghi = ghi.merge(gnhi, on = 'team', how = 'left')
ghi['game_homer_rate'] = ghi['games_homered_in'] / (ghi['games_homered_in'] + ghi['games_not_homered_in'])
ghi = ghi[['team', 'game_homer_rate']]
hr = hr.fillna(0)
hr = hr[hr['pre_score'] != hr['post_score']][['team', 'hr', 'pre_score', 'post_score']]
hr = hr.groupby(['team', 'hr'], as_index = False).sum()
hr['runs_scored'] = hr['post_score'] - hr['pre_score']
hr = hr.pivot(index = 'team', columns = 'hr', values = 'runs_scored')
hr = hr.fillna(0).reset_index()
hr['runs_scored'] = hr['homer'] + hr['non_homer']
hr['home_run_reliance'] = hr['homer'] / hr['runs_scored']
hr['percent_on_non_homer'] = hr['non_homer'] / hr['runs_scored']
hr = hr.merge(hpg, on = 'team')
hr = hr.merge(ghi, on = 'team')
hr = hr[['team', 'runs_scored', 'home_run_reliance', 'percent_on_non_homer', 'homer_per_game', 'game_homer_rate']]
for_merge = hr[['team', 'home_run_reliance', 'homer_per_game', 'game_homer_rate']]

In [ ]:
sus = data[['home_team', 'events']]
sus = sus.rename(columns = {'home_team' : 'team'})
sus['hr'] = sus.apply(homer, axis = 1)
sus = sus[sus['hr'] == 'homer']
sus = sus[['team', 'hr']]
sus = sus.groupby('team').count().reset_index()
games = data[['home_team', 'game_id']].drop_duplicates()
games = games.groupby('home_team').count().reset_index()
games = games.rename(columns = {'home_team' : 'team', 'game_id' : 'game_count'})
asus = data[['home_team', 'events', 'inning_topbot']]
asus = asus[asus['inning_topbot'] == 'Top']
asus['hr'] = asus.apply(homer, axis = 1)
asus = asus[asus['hr'] == 'homer']
asus = asus.rename(columns = {'home_team' : 'team', 'hr' : 'away_hr'})
asus = asus[['team', 'away_hr']]
asus = asus.groupby('team').count().reset_index()
sus = sus.merge(games, on = 'team')
sus['hr_rate'] = sus['hr'] / sus['game_count']
sus = sus.merge(asus, on = 'team')
sus['away_hr_rate'] = sus['away_hr'] / sus['game_count']
sus['team_home_run_rate'] = sus['hr_rate'] - sus['away_hr_rate']
sus = sus[['team', 'team_home_run_rate', 'away_hr_rate']]

In [ ]:
def  sv(df):
    if df['events'] == 'single':
        return 1
    elif df['events'] == 'double':
        return 2
    elif df['events'] == 'triple':
        return 3
    elif df['events'] == 'home_run':
        return 4
    elif df['events'] == 'walk':
        return 1
    else:
        return 0
top = data[data['inning_topbot'] == 'Top'][['game_id', 'away_team', 'events', 'home_team']]
top = top.rename(columns = {'away_team' : 'team', 'home_team' : 'opp'})
bot = data[data['inning_topbot'] == 'Bot'][['game_id', 'home_team', 'events', 'away_team']]
bot = bot.rename(columns = {'home_team' : 'team', 'away_team' : 'opp'})
bases = pd.concat([top, bot])
bases['bases'] = bases.apply(sv, axis = 1)
bases = bases.groupby(['team', 'opp', 'game_id'], as_index = False)['bases'].sum()
ba = bases.rename(columns = {'opp' : 'team', 'team' : 'opp', 'bases' : 'bases_allowed'})
full_bases = bases.merge(ba, on = ['team', 'opp','game_id'])

full_bases = full_bases.sort_values(['team', 'game_id'])

full_bases['bases_shift'] = (
    full_bases.groupby('team')['bases']
      .shift(1)
)

full_bases['bases_allowed_shift'] = (
    full_bases.groupby('team')['bases_allowed']
      .shift(1)
)

windows = [10, 30, 81]

for w in windows:

    # offensive rolling bases
    full_bases[f'bases_pg_{w}'] = (
        full_bases.groupby('team')['bases_shift']
          .rolling(w, min_periods=1)
          .mean()
          .reset_index(level=0, drop=True)
    )

    # defensive rolling bases allowed
    full_bases[f'bases_allowed_pg_{w}'] = (
        full_bases.groupby('team')['bases_allowed_shift']
          .rolling(w, min_periods=1)
          .mean()
          .reset_index(level=0, drop=True)
    )


full_bases['bases_pg_full'] = (
    full_bases.groupby('team')['bases_shift']
      .expanding()
      .mean()
      .reset_index(level=0, drop=True)
)

full_bases['bases_allowed_pg_full'] = (
    full_bases.groupby('team')['bases_allowed_shift']
      .expanding()
      .mean()
      .reset_index(level=0, drop=True)
)

full_bases['WAVE'] = (
    0.3 * full_bases['bases_pg_10'] +
    0.3 * full_bases['bases_pg_30'] +
    0.2 * full_bases['bases_pg_81'] +
    0.2 * full_bases['bases_pg_full']
)

full_bases['WAVE_allowed'] = (
    0.3 * full_bases['bases_allowed_pg_10'] +
    0.3 * full_bases['bases_allowed_pg_30'] +
    0.2 * full_bases['bases_allowed_pg_81'] +
    0.2 * full_bases['bases_allowed_pg_full']
)

opp_wave = full_bases[['team', 'game_id', 'WAVE_allowed']].rename(columns={
    'team': 'opp',
    'WAVE_allowed': 'opp_WAVE_allowed'
})

full_bases = full_bases.merge(
    opp_wave,
    on=['opp', 'game_id'],
    how='left'
)

full_bases = full_bases.fillna(0)

full_bases['offensive_edge'] = (
    full_bases['WAVE']
    -
    full_bases['opp_WAVE_allowed']
)

latest = full_bases.groupby('team', as_index = False)['game_id'].max()
latest_off = latest.merge(full_bases, on = ['team', 'game_id'])[['team', 'offensive_edge']]

In [ ]:
dt = data[['home_team', 'away_team', 'game_date', 'game_id', 'post_home_score', 'post_away_score']]
dt['fs'] = dt['post_home_score'] + dt['post_away_score']
gf = dt.groupby('game_id', as_index = False)['fs'].max()
gf = gf.merge(dt, on = ['game_id', 'fs']).drop_duplicates()[['home_team', 'away_team', 'game_date', 'game_id', 'post_home_score', 'post_away_score']]

In [ ]:
away = gf.rename(columns = {'away_team' : 'team', 'home_team' : 'opp', 'post_home_score' : 'ra', 'post_away_score' : 'rs'})
home = gf.rename(columns = {'home_team' : 'team', 'away_team' : 'opp', 'post_home_score' : 'rs', 'post_away_score' : 'ra'})
record = pd.concat([away, home]).sort_values(['team', 'game_date'])
record['win'] = (record['rs'] > record['ra']).astype('int')
record['loss'] = (record['rs'] < record['ra']).astype('int')
record = record[['opp', 'team', 'game_date', 'game_id', 'win', 'loss', 'rs', 'ra']]

In [ ]:
under3 = record.sort_values(['team', 'game_id'])
under3['under3'] = (under3['rs'] < 3).astype(int)
windows = [10, 30, 81, 162]

for w in windows:

    under3[f'under3_pct_{w}'] = (
        under3.groupby('team')['under3']
              .rolling(w, min_periods=1)
              .mean()
              .reset_index(level=0, drop=True)
    )

max3 = under3.groupby('team', as_index = False)['game_id'].max()
max3 = max3.merge(under3, on = ['team', 'game_id'])
max3['suppression_resistance'] = 1 - (max3['under3_pct_162'] * .2 + max3['under3_pct_81'] * .2 + max3['under3_pct_30'] * .3 + max3['under3_pct_10'] * .3)
max3 = max3[['team', 'suppression_resistance']]
sup_mean = max3['suppression_resistance'].mean()
std_sup = max3['suppression_resistance'].std()
max3['suppression_resistance'] = 1 + (((max3['suppression_resistance'] - sup_mean) / std_sup) * .15)

In [ ]:
import pandas as pd
import numpy as np

nf = record.sort_values(['team', 'game_id']).reset_index(drop=True)

windows = [10, 30, 81]

def compute_windows(group):
    group = group.copy()

    results = {f'roll_{w}_excl': [] for w in windows}
    full_excl = []

    for i in range(len(group)):
        opp = group.iloc[i]['opp']

        history = group.iloc[:i]

        # exclude games vs this opponent
        history_excl = history[history['opp'] != opp]

        # full record
        if len(history_excl) > 0:
            full_excl.append(history_excl['win'].mean())
        else:
            full_excl.append(np.nan)

        # rolling windows
        for w in windows:
            if len(history_excl) > 0:
                results[f'roll_{w}_excl'].append(
                    history_excl['win'].tail(w).mean()
                )
            else:
                results[f'roll_{w}_excl'].append(np.nan)

    for k in results:
        group[k] = results[k]

    group['full_excl'] = full_excl

    return group

nf = nf.groupby('team', group_keys=False).apply(compute_windows)
nf = nf.rename(columns = {'roll_10_excl' : 'rolling_10', 'roll_30_excl' : 'rolling_30', 'roll_81_excl' : 'rolling_81', 'full_excl' : 'full'}).fillna(0)

In [ ]:
def compute_current(group):
    group = group.copy()

    results = {f'roll_{w}_cur': [] for w in windows}
    full_excl = []

    for i in range(len(group)):
        opp = group.iloc[i]['opp']

        history = group.iloc[:i]

        if len(history) > 0:
            full_excl.append(history['win'].mean())
        else:
            full_excl.append(np.nan)

        for w in windows:
            if len(history) > 0:
                results[f'roll_{w}_cur'].append(
                    history['win'].tail(w).mean()
                )
            else:
                results[f'roll_{w}_cur'].append(np.nan)

    for k in results:
        group[k] = results[k]

    group['full_cur'] = full_excl

    return group

nf = nf.groupby('team', group_keys=False).apply(compute_current)
nf = nf.fillna(0)

In [ ]:
nf['rs_shift'] = nf.groupby('team')['rs'].shift(1)
nf['ra_shift'] = nf.groupby('team')['ra'].shift(1)
windows = [10, 30, 81]

for w in windows:

    nf[f'rs_{w}'] = (
        nf.groupby('team')['rs_shift']
          .rolling(w, min_periods=1)
          .sum()
          .reset_index(level=0, drop=True)
    )

    nf[f'ra_{w}'] = (
        nf.groupby('team')['ra_shift']
          .rolling(w, min_periods=1)
          .sum()
          .reset_index(level=0, drop=True)
    )

nf['rs_full'] = (
    nf.groupby('team')['rs_shift']
      .cumsum()
)

nf['ra_full'] = (
    nf.groupby('team')['ra_shift']
      .cumsum()
)

EXP = 1.83

for w in windows:

    nf[f'pyth_{w}'] = (
        nf[f'rs_{w}'] ** EXP
    ) / (
        (nf[f'rs_{w}'] ** EXP) +
        (nf[f'ra_{w}'] ** EXP)
    )

nf['pyth_full'] = (
    nf['rs_full'] ** EXP
) / (
    (nf['rs_full'] ** EXP) +
    (nf['ra_full'] ** EXP)
)

nf = nf.fillna(0)

In [ ]:
from numpy._core.fromnumeric import std
nf['strength'] = (nf['rolling_10']*.35 + nf['rolling_30']*.3 + nf['rolling_81']*.2 + nf['full']*.15)
nf['current_strength'] = nf['roll_10_cur']*.35 + nf['roll_30_cur']*.3 + nf['roll_81_cur']*.2 + nf['full_cur']*.15
nf['pyth_strength'] = (nf['pyth_10']*.35 + nf['pyth_30']*.3 + nf['pyth_81']*.2 + nf['pyth_full']*.15)
sos = nf.groupby('opp', as_index = False)[['strength', 'pyth_strength']].mean()
sos = sos.rename(columns = {'opp' : 'team', 'strength' : 'SOS', 'pyth_strength' : 'pyth_SOS'})
latest = nf.groupby('team', as_index = False)['game_id'].max()
current_strength = latest.merge(nf, on = ['team', 'game_id'])[['team', 'strength', 'current_strength', 'pyth_strength']]
master = current_strength.merge(sos, on = 'team')
mean_strength = master['strength'].mean()
mean_SOS = master['SOS'].mean()
mean_current_strength = master['current_strength'].mean()
mean_pyth_strength = master['pyth_strength'].mean()
mean_pyth_SOS = master['pyth_SOS'].mean()
std_strength = master['strength'].std()
std_SOS = master['SOS'].std()
std_current_strength = master['current_strength'].std()
std_pyth_strength = master['pyth_strength'].std()
std_pyth_SOS = master['pyth_SOS'].std()
master['norm_strength'] = 1 + (((master['strength'] - mean_strength) / std_strength) * .15)
master['norm_sos'] = 1 + (((master['SOS'] - mean_SOS) / std_SOS) * .15)
master['norm_current_strength'] = 1 + (((master['current_strength'] - mean_current_strength) / std_current_strength) * .15)
master['norm_pyth_strength'] = 1 + (((master['pyth_strength'] - mean_pyth_strength) / std_pyth_strength) * .15)
master['norm_pyth_sos'] = 1 + (((master['pyth_SOS'] - mean_pyth_SOS) / std_pyth_SOS) * .15)
master = master[['team', 'norm_strength', 'norm_sos', 'norm_current_strength', 'norm_pyth_strength', 'norm_pyth_sos']].rename(columns = {'norm_strength' : 'Strength', 'norm_sos' : 'SOS', 'norm_current_strength' : 'current', 'norm_pyth_strength' : 'pyth_Strength', 'norm_pyth_sos' : 'pyth_SOS'}).drop_duplicates().reset_index(drop = True)
master['Confidence'] = master['Strength'] + (master['SOS'] *.3)
master['pyth_Confidence'] = master['pyth_Strength'] + (master['pyth_SOS'] *.3)
mean_conf = master['Confidence'].mean()
std_conf = master['Confidence'].std()
mean_pyth_conf = master['pyth_Confidence'].mean()
std_pyth_conf = master['pyth_Confidence'].std()
master['Confidence'] = 1 + (((master['Confidence'] - mean_conf)/std_conf)*.15)
master['pyth_Confidence'] = 1 + (((master['pyth_Confidence'] - mean_pyth_conf) / std_pyth_conf) * .15)
master = master[['team', 'current', 'Strength', 'pyth_Strength', 'SOS', 'pyth_SOS', 'Confidence', 'pyth_Confidence']]
master['Confidence_Delta'] = master['Confidence'] - master['pyth_Confidence']
master = master.merge(latest_off, on = 'team')
master = master.merge(for_merge, on = 'team')
master = master.merge(sus, on = 'team')
master = master.merge(max3, on = 'team')
mean_edge = master['offensive_edge'].mean()
std_edge = master['offensive_edge'].std()
master['offensive_edge'] = 1 + (((master['offensive_edge'] - mean_edge) / std_edge) * .15)
master = master.drop_duplicates()
master['true_power'] = (master['offensive_edge'] + master['suppression_resistance'])/2
master = master[['team', 'current', 'Strength', 'pyth_Strength', 'SOS', 'pyth_SOS', 'Confidence', 'pyth_Confidence', 'Confidence_Delta', 'true_power', 'offensive_edge', 'suppression_resistance', 'home_run_reliance', 'homer_per_game', 'game_homer_rate', 'team_home_run_rate', 'away_hr_rate']]
master = master.sort_values('pyth_Confidence', ascending = False).reset_index(drop = True)

#GITHUB

In [ ]:
repo = "/content/mlb_metrics"

WAVE.to_csv(
    "/data/wave.csv",
    index=False
)

pave.to_csv(
    "/data/pave.csv",
    index=False
)

master.to_csv(
    "/data/confidence.csv",
    index=False
)